# Credit Card Fraud Detection

UTS 32513 - Advanced Data Analytics Algorithms
Assessment 2: Machine Learning Model Study / Project Implementation

This notebook builds a fraud detection system on the ULB Credit Card Fraud Detection dataset.

## 1. Setup

In [ ]:
import os
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, precision_recall_curve, confusion_matrix
)
from scipy.stats import loguniform, randint

try:
    import lightgbm as lgb
except ImportError:
    !pip install -q lightgbm
    import lightgbm as lgb

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Load Data

The dataset is downloaded directly from a public URL so the notebook is self-contained.

In [ ]:
DATA_URL = "https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv"
DATA_PATH = "creditcard.csv"

if not os.path.exists(DATA_PATH):
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print(df["Class"].value_counts())
print(df.head())

## 3. Temporal Train/Validation/Test Split

Fraud patterns change over time. Splitting by the `Time` column avoids future transactions leaking into training.

In [ ]:
df = df.sort_values("Time").reset_index(drop=True)
n = len(df)
train_end = int(n * 0.6)
val_end = int(n * 0.8)

train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]

feature_cols = [c for c in df.columns if c not in ["Time", "Class"]]

scaler = StandardScaler()

def make_xy(sub_df, fit_scaler=False):
    X = sub_df[feature_cols].copy()
    if fit_scaler:
        X["Amount"] = scaler.fit_transform(X[["Amount"]])
    else:
        X["Amount"] = scaler.transform(X[["Amount"]])
    y = sub_df["Class"].values
    return X, y

X_train, y_train = make_xy(train_df, fit_scaler=True)
X_val, y_val = make_xy(val_df)
X_test, y_test = make_xy(test_df)

print(f"Train: {X_train.shape}, frauds={y_train.sum()}")
print(f"Val:   {X_val.shape}, frauds={y_val.sum()}")
print(f"Test:  {X_test.shape}, frauds={y_test.sum()}")

## 4. Model

We use gradient boosted decision trees (LightGBM). The loss is binary cross-entropy. Class weights are applied because fraud is rare.

In [ ]:
def build_lgbm(class_weight="balanced", n_estimators=200, learning_rate=0.05,
               num_leaves=31, max_depth=-1, **kwargs):
    return lgb.LGBMClassifier(
        objective="binary",
        boosting_type="gbdt",
        class_weight=class_weight,
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        num_leaves=num_leaves,
        max_depth=max_depth,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
        **kwargs
    )

baseline = build_lgbm()
baseline.fit(X_train, y_train)
val_probs_baseline = baseline.predict_proba(X_val)[:, 1]
print("Baseline val AP:", average_precision_score(y_val, val_probs_baseline))

## 5. Hyperparameter Tuning

In [ ]:
param_dist = {
    "n_estimators": randint(100, 600),
    "learning_rate": loguniform(1e-2, 3e-1),
    "num_leaves": randint(20, 150),
    "max_depth": randint(3, 12),
    "min_child_samples": randint(10, 200),
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
}

search = RandomizedSearchCV(
    build_lgbm(),
    param_distributions=param_dist,
    n_iter=15,
    scoring="average_precision",
    cv=3,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1
)
search.fit(X_train, y_train)

best_model = search.best_estimator_
print("Best params:", search.best_params_)
print("Best CV AP:", search.best_score_)

val_probs = best_model.predict_proba(X_val)[:, 1]
test_probs = best_model.predict_proba(X_test)[:, 1]

## 6. Cost-Sensitive Threshold Optimization

A missed fraud is more expensive than a false alarm. We define cost ratios and pick the threshold that minimizes total cost on the validation set.

In [ ]:
COST_FN = 100.0  # cost of missing one fraud
COST_FP = 10.0   # cost of one false alarm

def total_cost(y_true, y_pred, cost_fn, cost_fp):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return fn * cost_fn + fp * cost_fp

thresholds = np.linspace(0.001, 0.999, 999)
costs = [total_cost(y_val, (val_probs >= t).astype(int), COST_FN, COST_FP) for t in thresholds]
best_idx = int(np.argmin(costs))
best_threshold = thresholds[best_idx]
best_cost = costs[best_idx]

print(f"Best threshold: {best_threshold:.4f}, cost: {best_cost:.0f}")

plt.figure(figsize=(6, 4))
plt.plot(thresholds, costs, label="Total cost")
plt.axvline(best_threshold, color="red", linestyle="--", label=f"Best = {best_threshold:.3f}")
plt.xlabel("Threshold")
plt.ylabel("Total cost")
plt.title("Cost vs. Threshold on Validation Set")
plt.legend()
plt.grid(True)
plt.show()

## 7. Evaluation on Test Set

In [ ]:
y_pred_test = (test_probs >= best_threshold).astype(int)

metrics = {
    "accuracy": accuracy_score(y_test, y_pred_test),
    "precision": precision_score(y_test, y_pred_test, zero_division=0),
    "recall": recall_score(y_test, y_pred_test, zero_division=0),
    "f1": f1_score(y_test, y_pred_test, zero_division=0),
    "auroc": roc_auc_score(y_test, test_probs),
    "average_precision": average_precision_score(y_test, test_probs),
    "total_cost": total_cost(y_test, y_pred_test, COST_FN, COST_FP),
}

for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

precision, recall, _ = precision_recall_curve(y_test, test_probs)
plt.figure(figsize=(6, 4))
plt.plot(recall, precision, label=f"AP = {metrics['average_precision']:.4f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve on Test Set")
plt.legend()
plt.grid(True)
plt.show()

## 8. Why These Results Matter

- Accuracy is near 99.9% but misleading on imbalanced data.
- AUROC and average precision show the model ranks frauds well.
- Threshold selection by cost reflects the real business trade-off.
- Temporal splitting makes the estimate realistic for production deployment.